# Data Quality & Validation — Pandera, DeepChecks, Evidently, Pointblank

This notebook validates the MNIST 1D dataset and trained model against the full quality stack:

| Tool | Purpose | Role |
|---|---|---|
| **Pandera** | Schema validation — feature ranges, label domain, null checks | Data Preparation |
| **DeepChecks** | Dataset integrity + train/test distribution suites | Evaluation |
| **Evidently** | Data drift + quality HTML reports → logged to MLflow/MinIO | Evaluation + Deployment |
| **Pointblank** | Interactive stakeholder-facing data quality reports | Evaluation |

> **Run this notebook after** `mnist1d_crisp_dm.ipynb` so `best_model.pt` and the dataset exist.


In [ ]:
import subprocess, sys
packages = ['pandera', 'deepchecks', 'evidently', 'pointblank', 'mlflow', 'boto3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + packages)
print('Ready.')


In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import mlflow
warnings.filterwarnings('ignore')

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000')
SEED = 42

with open('mnist1d_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = np.array(data['x'],      dtype=np.float32)
y_train = np.array(data['y'],      dtype=np.int64)
X_test  = np.array(data['x_test'], dtype=np.float32)
y_test  = np.array(data['y_test'], dtype=np.int64)

N_FEATURES = X_train.shape[1]   # 40
COLS = [f't{i}' for i in range(N_FEATURES)]

df_train = pd.DataFrame(X_train, columns=COLS)
df_train['label'] = y_train
df_test  = pd.DataFrame(X_test,  columns=COLS)
df_test['label']  = y_test

print(f'Train: {df_train.shape}  Test: {df_test.shape}')


## § 1 Pandera — Schema Validation

Pandera enforces contracts on DataFrames at code time or at runtime as a decorator. It has a concise Python API with no YAML, and native pandas, Polars, and PySpark support.


In [ ]:
import pandera as pa

# ── Define schema ────────────────────────────────────────────────────────
feature_checks = {c: pa.Column(float, pa.Check.between(-8.0, 8.0), nullable=False) for c in COLS}
feature_checks['label'] = pa.Column(int, pa.Check.isin(list(range(10))), nullable=False)

schema = pa.DataFrameSchema(
    feature_checks,
    coerce=True,
    strict=False,       # allow extra columns (future-proof)
)

# ── Run validation ────────────────────────────────────────────────────────
try:
    schema.validate(df_train, lazy=True)   # lazy=True collects all errors
    print(f'Train schema  PASS  ({len(df_train):,} rows)')
except pa.errors.SchemaErrors as e:
    print('Train schema  FAIL')
    print(e.failure_cases.head())

try:
    schema.validate(df_test, lazy=True)
    print(f'Test  schema  PASS  ({len(df_test):,} rows)')
except pa.errors.SchemaErrors as e:
    print('Test  schema  FAIL')
    print(e.failure_cases.head())


In [ ]:
# ── Class-based schema (alternative, supports type annotations) ──────────
class Mnist1DSchema(pa.DataFrameModel):
    label: pa.typing.Series[int] = pa.Field(isin=list(range(10)))

    class Config:
        coerce = True

validated_df = Mnist1DSchema.validate(df_train)
print(f'Class-based schema PASS  shape={validated_df.shape}')


## § 2 DeepChecks — Dataset Integrity & Train/Test Validation

DeepChecks runs suites of checks that are specific to ML: feature drift, label drift, label ambiguity, duplicate samples, and more. Results are interactive HTML reports.


In [ ]:
from deepchecks.tabular import Dataset
from deepchecks.tabular.suites import data_integrity, train_test_validation

train_ds = Dataset(df_train, label='label', cat_features=[])
test_ds  = Dataset(df_test,  label='label', cat_features=[])

print('Running data integrity suite on training set...')
integrity_result = data_integrity().run(train_ds)
integrity_result.save_as_html('deepchecks_integrity.html')
print('Saved: deepchecks_integrity.html')

print('Running train/test validation suite...')
tt_result = train_test_validation().run(train_ds, test_ds)
tt_result.save_as_html('deepchecks_train_test.html')
print('Saved: deepchecks_train_test.html')


In [ ]:
# Show inline summary
print('=== Data integrity: passed / failed checks ===')
for check_result in integrity_result.get_not_passed_checks():
    print(f'  FAIL  {check_result.get_header()}')
passed = len(integrity_result.get_passed_checks())
failed = len(integrity_result.get_not_passed_checks())
print(f'  {passed} passed  {failed} failed')

print()
print('=== Train/test: passed / failed checks ===')
for check_result in tt_result.get_not_passed_checks():
    print(f'  FAIL  {check_result.get_header()}')
passed2 = len(tt_result.get_passed_checks())
failed2 = len(tt_result.get_not_passed_checks())
print(f'  {passed2} passed  {failed2} failed')


In [ ]:
# Log DeepChecks reports to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment('mnist1d-quality')

with mlflow.start_run(run_name='deepchecks') as run:
    mlflow.log_metric('integrity_passed', passed)
    mlflow.log_metric('integrity_failed', failed)
    mlflow.log_metric('traintest_passed', passed2)
    mlflow.log_metric('traintest_failed', failed2)
    mlflow.log_artifact('deepchecks_integrity.html')
    mlflow.log_artifact('deepchecks_train_test.html')
    QUALITY_RUN_ID = run.info.run_id

print(f'DeepChecks reports logged — run {QUALITY_RUN_ID}')


## § 3 Evidently AI — Data Drift & Quality Reports

Evidently generates rich HTML reports and JSON snapshots. Snapshots can be pushed to the `evidently` container's workspace for persistent monitoring. Here we generate a drift report (train as reference, test as current data) and log it to MLflow.


In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.test_suite import TestSuite
from evidently.test_preset import DataStabilityTestPreset, NoTargetPerformanceTestPreset

# HTML report
report = Report(metrics=[DataQualityPreset(), DataDriftPreset()])
report.run(reference_data=df_train, current_data=df_test)
report.save_html('evidently_drift_report.html')
print('Saved: evidently_drift_report.html')

# Test suite (pass/fail assertions)
suite = TestSuite(tests=[DataStabilityTestPreset()])
suite.run(reference_data=df_train, current_data=df_test)
suite.save_html('evidently_tests.html')
print('Saved: evidently_tests.html')


In [ ]:
# Log to MLflow
with mlflow.start_run(run_id=QUALITY_RUN_ID):
    mlflow.log_artifact('evidently_drift_report.html')
    mlflow.log_artifact('evidently_tests.html')
    # Log pass/fail counts from the test suite
    results = suite.as_dict()
    n_passed = sum(1 for t in results.get('tests', []) if t.get('status') == 'SUCCESS')
    n_failed = sum(1 for t in results.get('tests', []) if t.get('status') == 'FAIL')
    mlflow.log_metric('evidently_tests_passed', n_passed)
    mlflow.log_metric('evidently_tests_failed', n_failed)
print(f'Evidently: {n_passed} tests passed  {n_failed} failed')
print(f'Reports logged to MLflow run {QUALITY_RUN_ID}')


In [ ]:
# ── Optional: push snapshot to the Evidently monitoring service ──────────
# Requires the 'evidently' service stack to be running (./scripts/start.sh evidently)
EVIDENTLY_SERVICE_URL = os.environ.get('EVIDENTLY_SERVICE_URL', 'http://localhost:8001')

try:
    from evidently.ui.workspace import Workspace
    ws = Workspace(url=EVIDENTLY_SERVICE_URL)
    project = ws.create_project('MNIST 1D monitoring')
    project.save()
    ws.add_run(project.id, report)
    print(f'Snapshot pushed to Evidently service at {EVIDENTLY_SERVICE_URL}')
except Exception as e:
    print(f'Evidently service not reachable ({e!r}) — report is still in MLflow.')


## § 4 Pointblank — Interactive Data Quality Reports

[Pointblank](https://github.com/posit-dev/pointblank) is a 2024 release from Posit (makers of RStudio/Tidyverse). It produces interactive, self-contained HTML validation reports for pandas and Polars DataFrames, designed for stakeholder-facing quality reviews.


In [ ]:
import pointblank as pb

# ── Build a validation plan ───────────────────────────────────────────────
v = (
    pb.Validate(df_train, tbl_name='MNIST1D-train', label='Data Quality Check')
    .col_vals_not_null(columns=COLS + ['label'])
    .col_vals_between(columns=COLS, left=-8.0, right=8.0)
    .col_vals_in_set(columns=['label'], set=list(range(10)))
    .col_count_match(count=N_FEATURES + 1)   # 40 features + label
    .row_count_match(count=4000)
    .interrogate()
)

print(v)


In [ ]:
# Export Pointblank report to HTML and log to MLflow
pb_html = 'pointblank_report.html'
v.get_tabular_report().write_html(pb_html)
print(f'Saved: {pb_html}')

with mlflow.start_run(run_id=QUALITY_RUN_ID):
    mlflow.log_artifact(pb_html)
print(f'Pointblank report logged to MLflow run {QUALITY_RUN_ID}')


## Summary

| Run this notebook gets you | Stored in |
|---|---|
| Pandera schema pass/fail | console |
| DeepChecks integrity HTML | MLflow artifact (MinIO) |
| DeepChecks train/test HTML | MLflow artifact (MinIO) |
| Evidently drift report HTML | MLflow artifact (MinIO) + Evidently service |
| Evidently test suite HTML | MLflow artifact (MinIO) |
| Pointblank interactive report HTML | MLflow artifact (MinIO) |

The same validation logic runs inside the Dagster pipeline (`services/dagster/pipelines/mnist1d_pipeline.py`) automatically when you trigger `mnist1d_full_pipeline` from the Dagster UI (`http://localhost:3000`).
